In [2]:
!uv add pandas numpy python-dotenv langchain langchain-core langchain-community langchain-google-genai langchain-openai openai chromadb langchain-chroma langchain-huggingface faiss-cpu pypdf pymupdf 

Resolved 125 packages in 5.46s
 Downloaded faiss-cpu
 Downloaded pymupdf
 Downloaded google-genai
 Downloaded openai
Prepared 10 packages in 29.51s
Installed 117 packages in 48.80s
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anyio==4.14.2
 + attrs==26.1.0
 + bcrypt==5.0.0
 + build==1.5.0
 + certifi==2026.7.22
 + cffi==2.1.0
 + charset-normalizer==3.4.9
 + chromadb==1.5.9
 + click==8.4.2
 + cryptography==49.0.0
 + distro==1.9.0
 + durationpy==0.10
 + faiss-cpu==1.14.3
 + filelock==3.32.2
 + filetype==1.2.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + fsspec==2026.7.0
 + google-auth==2.56.2
 + google-genai==2.16.0
 + googleapis-common-protos==1.75.0
 + greenlet==3.5.4
 + grpcio==1.83.0
 + h11==0.16.0
 + hf-xet==1.5.2
 + httpcore==1.0.9
 + httptools==0.8.0
 + httpx==0.28.1
 + httpx-sse==0.4.3
 + huggingface-hub==1.26.0
 + idna==3.18
 + importlib-resources==7.1.0
 + jiter==0.16.0
 + jsonpatch==1.33
 + jsonpointe

In [3]:
!uv add chromadb langchain-chroma groq langchain-groq

Resolved 127 packages in 8.19s
Prepared 2 packages in 409ms
Installed 2 packages in 887ms
 + groq==0.37.1
 + langchain-groq==1.1.3


In [79]:
from dotenv import load_dotenv
import os

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

load_dotenv()

True

In [80]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
# Create Geminigpt model
gmini_llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash', 
    temperature=0
)

In [81]:
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
# Create ChatGPT model
gpt_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [82]:
from dotenv import load_dotenv
import os

from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

load_dotenv()

# Create and test a Groq LLM
groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY"),
)

prompt = [HumanMessage(content="Say hello in one short sentence.")]
response = groq_llm.invoke(prompt)
print(response.content)


Hello.


### Steps 
1.  load the pdfs
2. chunks 
3. embeddings
4. retriver search 
5. llm add
6. add tools for search or calculations
7. generations the result 


In [83]:
!uv add langchain-text-splitters

Resolved 127 packages in 3ms
Checked 125 packages in 52ms


In [84]:
!uv pip list


Package                                  Version
---------------------------------------- -----------
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.3
aiosignal                                1.4.0
annotated-doc                            0.0.5
annotated-types                          0.8.0
anyio                                    4.14.2
asttokens                                3.0.2
attrs                                    26.1.0
bcrypt                                   5.0.0
build                                    1.5.0
certifi                                  2026.7.22
cffi                                     2.1.0
charset-normalizer                       3.4.9
chromadb                                 1.5.9
click                                    8.4.2
colorama                                 0.4.6
comm                                     0.2.3
cryptography                             49.0.0
debugpy                                  1.8

In [85]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load all PDFs from the "pdfs" directory
loader = DirectoryLoader(
    path="pdfs",
    glob="**/*.pdf",        # Load PDFs recursively
    loader_cls=PyPDFLoader,
    show_progress=True,
)


In [86]:

documents = loader.load()

print(f"Loaded {len(documents)} pages.")

100%|██████████| 14/14 [00:20<00:00,  1.44s/it]

Loaded 105 pages.


In [87]:
# Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = text_splitter.split_documents(documents)

# Add PDF filename to every chunk before embedding.
# This helps queries like "Jaisalmer" match the Jaisalmer PDF even if the page text is sparse.
for chunk in chunks:
    source_name = os.path.basename(chunk.metadata.get("source", ""))
    page = chunk.metadata.get("page", "")
    chunk.page_content = f"Source PDF: {source_name}\nPage: {page}\n{chunk.page_content}"

print(f"Created {len(chunks)} chunks.")

Created 112 chunks.


In [88]:
# Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2"
)

In [89]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


In [90]:
# Vector store
from langchain_chroma import Chroma

COLLECTION_NAME = "travel_pdf_rag_experiment"

# Recreate only this notebook collection so old/stale vectors do not affect answers.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)
vectorstore.reset_collection()
vectorstore.add_documents(chunks)


['93fbd9ee-fc56-448f-b9b7-80dd706694d8',
 '9ab272ad-a10f-4605-aae1-d7c51d369dda',
 '97d36d96-97a0-470f-b19a-7944b5f74ec8',
 '0cc31749-44b0-4c44-83d9-0bb34cd80d08',
 '78c0fb14-7b68-4713-97f8-4993bdb116e4',
 '70688d3e-d0c4-48da-9ef4-fa6fc386eb7e',
 '883cd92e-4b31-4de3-b7c5-155a77b1d272',
 'dcf272b2-fabd-4544-a262-fb26f0e3ba53',
 '22ec0f67-a4f2-4152-a286-70da5811bd57',
 '50235011-7db0-4136-81e1-9e27906f3c04',
 '1aca7d76-4539-4ff6-97f1-220ed5f08cb7',
 '2a33ee3f-3a17-436d-9b91-adcc880be842',
 'cefa0340-4719-49cd-bb3f-ffd27b36d839',
 '966c3768-f3e8-4b7e-8e32-36782a29e132',
 'e7a84009-002e-4e91-88f4-4b7971d107a4',
 'a46e5090-7a49-4e99-9290-637908284a74',
 '7d54ce2b-6b1f-4f51-a9ed-9f4d07f8067f',
 'cbd9b25c-c10e-48f0-bd2e-8d498b08831b',
 '6544d9e2-b44c-4599-8a94-a55be3736fe4',
 'b06edbb7-d44d-47d6-9157-1f01f924208c',
 '443318cd-cb82-465e-9d5f-19b7e7576650',
 '52bd498d-6d86-4ade-9094-cfcedc7400cf',
 '99ffdbf9-dc6b-4de2-a002-93e4e52c7496',
 'c8f004bc-7d63-400a-b386-a6dc994b6ce6',
 'bd150df8-b474-

In [91]:

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 30},
)

In [92]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000202941491C0>, search_type='mmr', search_kwargs={'k': 8, 'fetch_k': 30})

In [93]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [94]:
# -------------------------------------
# Groq LLM
# -------------------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

# -------------------------------------
# Prompt
# -------------------------------------
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer ONLY from the provided context.
If part of the question is not present in the context, clearly say that part is not provided, then answer the parts that are present.
For example, if the user asks for an upcoming date but the PDFs do not contain dates, say the upcoming date is not provided in the PDFs and still summarize the itinerary/package from context.
If no relevant context is found at all, say:
"I couldn't find that information in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""
)



In [95]:
def normalize_question_for_retrieval(question):
    remove_terms = [
        "with upcoming date",
        "upcoming date",
        "upcoming dates",
        "future date",
        "future dates",
        "available date",
        "available dates",
    ]
    cleaned = question
    for term in remove_terms:
        cleaned = cleaned.replace(term, "")
        cleaned = cleaned.replace(term.title(), "")
    return " ".join(cleaned.split())


def format_docs(docs):
    formatted_docs = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page", "unknown")
        formatted_docs.append(f"[{source}, page {page}]\n{doc.page_content}")
    return "\n\n".join(formatted_docs)


In [96]:
# RAG pipeline using LCEL
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(normalize_question_for_retrieval(x["question"]))
        ),
        "question": lambda x: x["question"],
    }
    | prompt
    | llm
)


In [97]:
# Debug retrieved sources first, then ask question
debug_docs = retriever.invoke("Jaisalmer itinerary package")
print("Retrieved sources:")
for doc in debug_docs:
    print("-", os.path.basename(doc.metadata.get("source", "unknown")), "page", doc.metadata.get("page"))
response = rag_chain.invoke(
    {
        "question": "Tell me about the Jaisalmer itinerary package."
    }
)
print(response.content)

Retrieved sources:
- Jaisalmer Itinerary  2.pdf page 1
- Jaisalmer Itinerary  2.pdf page 6
- Jaisalmer Itinerary  2.pdf page 0
- Udaipur itinerary .pdf page 3
- 3 night 4 days Manali & kasol (5).pdf.pdf page 2
- DESTINO Udaipur Kumbhalgarh Brochure-1.pdf.pdf page 17
- Mussoorie to rishikesh via Dehradun itinerary .pdf.pdf.pdf page 1
- Chakrata 1N 2D.pdf page 0
The Jaisalmer itinerary package is for 2 nights and 3 days. It includes hotel, transport, and guided adventures. The itinerary is as follows:
- Day 0: Gurugram - Jaisalmer
- Day 1: Arrival at the campsite and desert safari
- Day 2: Visit to Longewala, Tanot, and Kuldhara
- Day 3: Jaisalmer Fort and Gadisar Lake

The package cost excludes entrance fees to monuments, medical expenses, tips, laundry, liquor, wine, mineral water, telephone charges, camera fees, and items of personal nature. 

You can book the package by contacting +91 9871536953 or +91 8448702976.


In [98]:
debug_docs = retriever.invoke(normalize_question_for_retrieval("Tell me about Kasol itinerary package. with upcoming date"))
print("Retrieved sources:")
for doc in debug_docs:
    print("-", os.path.basename(doc.metadata.get("source", "unknown")), "page", doc.metadata.get("page"))

response = rag_chain.invoke(
    {
        "question": "Tell me about Kasol itinerary package. with upcoming date"
    }
)
print(response.content)

Retrieved sources:
- 3 night 4 days Manali & kasol (5).pdf.pdf page 0
- 2N 3D kasol kheerganga .pdf page 4
- 2N 3D kasol kheerganga .pdf page 1
- 2N 3D kasol kheerganga .pdf page 0
- Destino Kasol Kheerganga Brochure 2024.pdf.pdf page 16
- Manali - Kasol - kheerganga.pdf page 1
- Manali - Kasol - kheerganga.pdf page 0
- 2N 3D kasol kheerganga .pdf page 5
The upcoming date is not provided in the context. 

The Kasol itinerary package includes a 2N 3D tour, with the following details:
- Departure from Delhi at 10:30 PM
- Arrival in Kasol, check-in to scenic camps, and exploration of Kasol's Mall Road and Manikaran
- Bonfire and DJ night at the camp
- Breakfast and dinner provided
- Sightseeing of Kasol and Manikaran
- Transfer from Delhi to Kasol and back

Additionally, there's a 3-night 4-day Manali and Kasol tour that includes:
- Local sightseeing in Manali, including Hadimba Devi Temple, Van Vihar, and Club House
- Visit to Solang Valley for adventure activities
- Head towards Kasol, 